GAN由两个神经网络组成：

**生成器：**

以随机分布作为输入（通常是高斯分布），并输出一些数据（通常
是图像）。你可以将随机输入视为要生成的图像的潜在表征（即编
码）。因此你可以看到，生成器提供的功能与变分自动编码器中的解码器相同，并且可以使用相同的方式来生成新图像（只需馈入一些高斯噪声，就会输出一个新图片）。

**判别器：**

输入从生成器得到的伪图像或从训练集中得到的真实图像，并且必
须猜测输入图像是伪图像还是真实图像。


在训练过程中，生成器和判别器有相反的目标：判别器试图从真实图像中分辨出虚假图像，而生成器则试图产生看起来足够真实的图像来
欺骗判别器。由于GAN由不同目标的两个网络组成，因此无法像常规神经网络一样对其进行训练。每个训练迭代都分为两个阶段：

·在第一阶段，我们训练判别器。从训练集中采样一批真实图像，再加上用生成器生成的相等数量的伪图像组成训练批次。对于伪图像，将标签设置为0；对于真实图像，将标签设置为1，并使用二元交叉熵损失在该被标签的批次上对判别器进行训练。重要的是，在这个阶段反向传播只能优化判别器的权重。

·在第二阶段，我们训练生成器。首先使用它来生成另一批伪图像，然后再次使用判别器来判断图像是伪图像还是真实图像。在这个批
次中我们不添加真实图像，并且所有标签都设置为1（真实）：换句话说，我们希望生成器能产生判别器会（错误地）认为是真实的图像！至关重要的是，在此步骤中，判别器的权重会被固定，因此反向传播只会影响生成器的权重。


In [ ]:
import keras
import tensorflow as tf
codings_size = 30 
generator = keras.models.Sequential([ 
    keras.layers.Dense(100, activation="selu", input_shape=[codings_size]), 
    keras.layers.Dense(150, activation="selu"), 
    keras.layers.Dense(28 * 28, activation="sigmoid"), 
    keras.layers.Reshape([28, 28]) 
]) 
discriminator = keras.models.Sequential([ 
    keras.layers.Flatten(input_shape=[28, 28]), 
    keras.layers.Dense(150, activation="selu"), 
    keras.layers.Dense(100, activation="selu"), 
    keras.layers.Dense(1, activation="sigmoid") 
]) 
gan = keras.models.Sequential([generator, discriminator])

discriminator.compile(loss="binary_crossentropy", optimizer="rmsprop") 
discriminator.trainable = False 
gan.compile(loss="binary_crossentropy", optimizer="rmsprop") 

batch_size = 32 
dataset = tf.data.Dataset.from_tensor_slices(X_train).shuffle(1000) 
dataset = dataset.batch(batch_size, drop_remainder=True).prefetch(1)


def train_gan(gan, dataset, batch_size, codings_size, n_epochs=50): 
    generator, discriminator = gan.layers 
    for epoch in range(n_epochs): 
        for X_batch in dataset: 
            # phase 1 - training the discriminator 
            noise = tf.random.normal(shape=[batch_size, codings_size]) 
            generated_images = generator(noise) 
            X_fake_and_real = tf.concat([generated_images, X_batch], axis=0) 
            y1 = tf.constant([[0.]] * batch_size + [[1.]] * batch_size) 
            discriminator.trainable = True 
            discriminator.train_on_batch(X_fake_and_real, y1) 
            # phase 2 - training the generator 
            noise = tf.random.normal(shape=[batch_size, codings_size]) 
            y2 = tf.constant([[1.]] * batch_size) 
            discriminator.trainable = False 
            gan.train_on_batch(noise, y2) 
train_gan(gan, dataset, batch_size, codings_size) 

- 在第一阶段，我们将高斯噪声馈送到生成器以生成伪图像，然后通过合并相等数量的真实图像来组成这一批次。对于伪图像，目标y1被
设置为0；对于真实图像，目标y1被设置为1。然后我们在这个批次上进行判别器训练。注意我们将判别器的可训练属性设置为True：这只是为了消除Keras注意到在编译模型时可训练属性为True，但是现在为False时显示的警告（反之亦然）。

- 在第二阶段，我们向GAN馈入一些高斯噪声。它的生成器首先会
生成伪图像，然后判别器会尝试猜测这些图像是伪图像还是真实图像。我们希望判别器相信伪图像是真实的，因此将目标y2设置为1。请注意，我们再次将可训练属性设置为False，以避免再次发出警告。


In [5]:
#深度卷积GAN（DCGAN）
#·用跨步卷积（在判别器中）和转置卷积（在生成器中）替换所有池化层。
#·除生成器的输出层和判别器的输入层外，在生成器和判别器中都使用批量归一化。
#·删除全连接的隐藏层以获得更深的架构。
#·对生成器中的所有层使用ReLU激活函数，除了输出层应该使用tanh。
#·对判别器中的所有层使用leaky ReLU激活函数。

codings_size = 100 
generator = keras.models.Sequential([ 
    keras.layers.Dense(7 * 7 * 128, input_shape=[codings_size]), 
    keras.layers.Reshape([7, 7, 128]), 
    keras.layers.BatchNormalization(), 
    keras.layers.Conv2DTranspose(64, kernel_size=5, strides=2, padding= "same", 
                             activation="selu"), 
    keras.layers.BatchNormalization(), 
    keras.layers.Conv2DTranspose(1, kernel_size=5, strides=2, padding= "same", 
                              activation="tanh") 
]) 
discriminator = keras.models.Sequential([ 
    keras.layers.Conv2D(64, kernel_size=5, strides=2, padding="same", 
                        activation=keras.layers.LeakyReLU(0.2), 
                        input_shape=[28, 28, 1]), 
    keras.layers.Dropout(0.4), 
    keras.layers.Conv2D(128, kernel_size=5, strides=2, padding="same", 
                        activation=keras.layers.LeakyReLU(0.2)), 
    keras.layers.Dropout(0.4), 
    keras.layers.Flatten(), 
    keras.layers.Dense(1, activation="sigmoid") 
]) 
gan = keras.models.Sequential([generator, discriminator]) 

In [ ]:
X_train = X_train.reshape(-1, 28, 28, 1) * 2. - 1. # reshape and rescale 
